In [3]:
import pandas as pd

df = pd.read_excel(r"C:\Users\Lenovo\Downloads\Massnahmenliste Gesamt AG_2025-11-20.xlsx", header=1)
df.head()

,Unnamed: 0,NR,Bestandser-\nhebungsbericht,Plan\nBautechnik,Plan\nEM-Technik,Strecke,RFB,Kilometer,Anzahl\nFahrspuren,Position,...,Mess-\nquerschnitt,Mess-\nquerschnitt\nNEU,Video,Video\nNEU,UDE,UDE\nNEU,zuständige\nABM,zuständige\nInstand-\nhaltung,zuständiger\nCN.as Koordinator,Anmerkungen
0,NaN,A01+S33 - 1,--,--,--,A01,2,15.696,--,"48.175735, 16.150301",...,--,--,--,--,--,--,St. Pölten,Hr. Stastny,Hr. Bruckmüller,--
1,NaN,A01+S33 - 2,1,41-A01001,13-0101,A01,1,15.896,2,"48.175938, 16.149732",...,2x 8+1,2x 8+1,--,--,--,--,St. Pölten,Hr. Stastny,Hr. Bruckmüller,FVE-Standort
2,NaN,A01+S33 - 3,2,41-A01001,13-0101,A01,2,15.897,2,"48.175621, 16.149894",...,2x 8+1,2x 8+1,--,--,--,--,St. Pölten,Hr. Stastny,Hr. Bruckmüller,FVE-Standort
3,NaN,A01+S33 - 4,--,--,--,A01,1\n,26.839,--,"48.161520, 16.012832",...,--,--,--,--,--,--,St. Pölten,Hr. Stastny,Hr. Bruckmüller,--
4,NaN,A01+S33 - 5,3,41-A01002,13-0102,A01,1,26.832,2,"48.161512, 16.012905",...,2x 8+1,2x 8+1,--,--,--,--,St. Pölten,Hr. Stastny,Hr. Bruckmüller,FVE-Standort


In [ ]:
import glob
import os

pdf_base_path = r"C:\Users\Lenovo\Downloads\LG13-0101 ff Standortschemen EM\LG13-0101 ff Standortschemen EM"
image_base_path = r"C:\Users\Lenovo\Downloads\LG13-0101 ff Standortschemen EM\LG13-0101 ff Standortschemen EM"
# matching_files = []
# for file in glob.glob(os.path.join(pdf_base_path, "**", "*13-0101*"), recursive=True):
#     if os.path.isfile(file):
#         matching_files.append(file)


all_files = glob.glob(os.path.join(pdf_base_path, "*/*"), recursive=True)

In [ ]:
import fitz
from PIL import Image

file_ids = df['Plan\nEM-Technik']
NEEDLES = ["ÜBERSICHTSCHEMATA", "LEGENDE Datenanbindung"]

pdf_path = glob.glob(os.path.join(pdf_base_path, "**", "*.pdf"), recursive=True)
for pdf in pdf_path:
    print(pdf)
    if pdf:
        doc = fitz.open(pdf)
        temp_doc = fitz.open()
        top_left_global = None
        right_boundary_global = None
        for src_page in doc:

            src_rect = src_page.rect  # source page rect
            w, h = src_rect.br  # save its width, height
            src_rot = src_page.rotation  # save source rotation
            src_page.set_rotation(0)  # set rotation to 0 temporarily
            page = temp_doc.new_page(width=w, height=h)  # make output page
            page.show_pdf_page(  # insert source page
                page.rect,
                doc,
                src_page.number,
                rotate=-src_rot,  # use reversed original rotation
            )

            rects = []
            for needle in NEEDLES:
                if needle == "ÜBERSICHTSCHEMATA":
                    hits = page.search_for(needle)
                    top_left_global = hits[0]
                    rects.extend(hits)
                elif needle == "LEGENDE Datenanbindung":
                    hits = page.search_for(needle)
                    right_boundary_global = hits[0]
                    rects.extend(hits)
            if not rects:
                continue
            # rec_draw = fitz.Rect(right_boundary_global.x, top_left_global.y, top_left_global.x, right_boundary_global.y)
            # Rect(35.85009765625, 666.9404296875, 786.1331787109375, 2113.69580078125)
            rec_draw = fitz.Rect(top_left_global.top_left, right_boundary_global.bottom_left.x, page.rect.height)            
            page.set_cropbox(rec_draw)
            pix = page.get_pixmap()
            pix.save(os.path.join(image_base_path, os.path.splitext(os.path.basename(pdf))[0]) + ".png")

                # mode = "RGB" if pix.alpha == 0 else "RGBA" # Determine the correct mode
                # img_data = pix.samples
                # width = pix.width
                # height = pix.height
                # pil_image = Image.frombytes(mode, [width, height], img_data)
                
                # break


        #     all_ids = df[df['Plan\nEM-Technik'] == id]
        #     useful_columns = ['VBA\nBetriebsmittel', 'VBA\nBetriebsmittel\nNEU', 'Video', 'Video\nNEU', 'UDE', 'UDE\nNEU']
        #     new_df = all_ids.loc[:, useful_columns]
        #     new_df.replace('--', pd.NA, inplace=True)
        #     if new_df.isna().any().any():
        #         pass
        #     else:
        #         pass
        #         break
        # else:
        #     continue
        

In [ ]:
import fitz
from PIL import Image

file_ids = df['Plan\nEM-Technik']
NEEDLES = ["ÜBERSICHTSCHEMATA", "LEGENDE Datenanbindung"]

for id in file_ids.unique():
    if id not in ['', 'nan', '--']:
        pdf_path = glob.glob(os.path.join(pdf_base_path, "**", f"*{id}*.pdf"), recursive=True)
        print(pdf_path)
        if pdf_path:
            doc = fitz.open(pdf_path[0])
            temp_doc = fitz.open()
            top_left_global = None
            right_boundary_global = None

            for src_page in doc:
                src_rect = src_page.rect  # source page rect
                w, h = src_rect.br  # save its width, height
                src_rot = src_page.rotation  # save source rotation
                src_page.set_rotation(0)  # set rotation to 0 temporarily
                page = temp_doc.new_page(width=w, height=h)  # make output page
                page.show_pdf_page(  # insert source page
                    page.rect,
                    doc,
                    src_page.number,
                    rotate=-src_rot,  # use reversed original rotation
                )

            rects = []
            for needle in NEEDLES:
                if needle == "ÜBERSICHTSCHEMATA":
                    hits = page.search_for(needle)
                    top_left_global = hits[0]
                    rects.extend(hits)
                elif needle == "LEGENDE Datenanbindung":
                    hits = page.search_for(needle)
                    right_boundary_global = hits[0]
                    rects.extend(hits)
                if not rects:
                    continue
                # rec_draw = fitz.Rect(right_boundary_global.x, top_left_global.y, top_left_global.x, right_boundary_global.y)
            rec_draw = fitz.Rect(top_left_global.top_left, right_boundary_global.bottom_left.x, page.rect.height)            
            page.set_cropbox(rec_draw)
            pix = page.get_pixmap()
            mode = "RGB" if pix.alpha == 0 else "RGBA" # Determine the correct mode
            img_data = pix.samples
            width = pix.width
            height = pix.height
            pil_image = Image.frombytes(mode, [width, height], img_data)
            
            # print(pil_image.size)
            pil_image.save(os.path.join(image_base_path, os.path.splitext(os.path.basename(pdf))[0]) + ".png")
            break

        #     all_ids = df[df['Plan\nEM-Technik'] == id]
        #     useful_columns = ['VBA\nBetriebsmittel', 'VBA\nBetriebsmittel\nNEU', 'Video', 'Video\nNEU', 'UDE', 'UDE\nNEU']
        #     new_df = all_ids.loc[:, useful_columns]
        #     new_df.replace('--', pd.NA, inplace=True)
        #     if new_df.isna().any().any():
        #         pass
        #     else:
        #         pass
        #         break
        # else:
        #     continue
        

['C:\\Users\\Lenovo\\Downloads\\LG13-0101 ff Standortschemen EM\\LG13-0101 ff Standortschemen EM\\A01_S33\\A01_05984_VBA_302500869_LG13-0101_SH_E_E_V00.pdf']
C:\Users\Lenovo\Downloads\LG13-0101 ff Standortschemen EM\LG13-0101 ff Standortschemen EM\**\*13-0101*.pdf
C:\Users\Lenovo\Downloads\LG13-0101 ff Standortschemen EM\LG13-0101 ff Standortschemen EM\S33_05984_VBA_302500869_LG13-0122_SH_E_E_V00.png


In [36]:
new_df.isna()
# any().any()

,VBA\nBetriebsmittel,VBA\nBetriebsmittel\nNEU,Video,Video\nNEU,UDE,UDE\nNEU
1,True,True,True,True,True,True
2,True,True,True,True,True,True


In [18]:
all_ids.columns

Index(['Unnamed: 0', 'NR', 'Bestandser-\nhebungsbericht', 'Plan\nBautechnik',
       'Plan\nEM-Technik', 'Strecke', 'RFB', 'Kilometer', 'Anzahl\nFahrspuren',
       'Position', 'ID_NAME_VMIS2', 'ID_NAME_VMIS1', 'Montageart',
       'VBA\nBetriebsmittel', 'VBA\nBetriebsmittel\nNEU', 'Mess-\nquerschnitt',
       'Mess-\nquerschnitt\nNEU', 'Video', 'Video\nNEU', 'UDE', 'UDE\nNEU',
       'zuständige\nABM', 'zuständige\nInstand-\nhaltung',
       'zuständiger\nCN.as Koordinator', 'Anmerkungen'],
      dtype='object')

In [103]:
import fitz  # pip install pymupdf

IN_PDF  = r"C:\Users\Lenovo\Downloads\LG13-0101 ff Standortschemen EM\LG13-0101 ff Standortschemen EM\A02\A02_05984_VBA_302500869_LG13-0288_SH_E_E_V00.pdf"
OUT_PDF = "cropped.pdf"
out_image = "cropped_image.png"


# how much extra padding around detected text (points). Increase if you want more context.
PAD_X, PAD_Y = 30, 20

doc = fitz.open(IN_PDF)
out = fitz.open()

top_left_global = None
right_boundary_global = None
for page in doc:
    rects = []
    for needle in NEEDLES:
        # rectangles where this text occurs (case-sensitive by default)
        if needle == "ÜBERSICHTSCHEMATA":
            hits = page.search_for(needle)
            top_left_global = hits[0].top_left
            print("top_left_global", top_left_global)
            rects.extend(hits)
        elif needle == "übertragen":
            hits = page.search_for(needle)
            right_boundary_global = hits[0].bottom_left
            rects.extend(hits)
        
    # break

    if not rects:
        continue
    
    
    # draw one rectangle per hit
    rec_draw = fitz.Rect(right_boundary_global.x, top_left_global.y, top_left_global.x, right_boundary_globa9l.y)
    print(rec_draw)
    # annot = page.add_rect_annot(
    #     rec_draw
    # )   # annotation rectangle
    # annot.set_colors(stroke=(1, 0, 0))  # red (RGB 0..1)
    # annot.set_border(width=1.5)
    # annot.update()


    # add padding and clamp to page bounds
    # crop = fitz.Rect(
    #     max(page.rect.x0, crop.x0 - PAD_X),
    #     max(page.rect.y0, crop.y0 - PAD_Y),
    #     min(page.rect.x1, crop.x1 + PAD_X),
    #     min(page.rect.y1, crop.y1 + PAD_Y),
    # )

    # Create a new page with the cropped content rendered into it
    # new_page = out.new_page(width=rec_draw.width, height=rec_draw.height)
    # new_page.show_pdf_page(
    #     fitz.Rect(0, 0, rec_draw.width, rec_draw.height),
    #     doc,
    #     page.number,
    #     clip=rec_draw
    # )
    page.set_cropbox(rec_draw)
    pix = page.get_pixmap()
    pix.save(out_image)

# out.save(OUT_PDF)
doc.save("abc.pdf")
out.close()
doc.close()
print("Wrote", OUT_PDF)


top_left_global Point(786.1331787109375, 666.9404296875)
Rect(35.85009765625, 666.9404296875, 786.1331787109375, 2113.69580078125)
Wrote cropped.pdf


In [83]:
top_left_global, right_boundary_global

(Point(786.1331787109375, 666.9404296875), 687.5650024414062)

In [85]:
fitz.Rect(top_left_global.x, top_left_global.y, right_boundary_global, )


Rect(786.1331787109375, 666.9404296875, 687.5650024414062, 65654.0)

In [48]:
hits

[Rect(786.1331787109375, 1337.5009765625, 805.9330444335938, 1647.6708984375)]

In [56]:
for rect in rects:
    rect.top_left 

[Rect(786.1331787109375, 666.9404296875, 805.9330444335938, 894.6226806640625),
 Rect(786.1331787109375, 1337.5009765625, 805.9330444335938, 1647.6708984375)]

In [70]:
doc = fitz.open(IN_PDF)
page = doc[0]
page.rect.width


3153.80078125

In [51]:
607,23px 
1860,18px

Rect(756.1331787109375, 646.9404296875, 835.9330444335938, 842.052001953125)

In [149]:
# doc = fitz.open(r"C:\Users\Lenovo\Downloads\LG13-0101 ff Standortschemen EM\LG13-0101 ff Standortschemen EM\S01\S01_05984_VBA_302500869_LG13-0521_SH_E_E_V00.pdf")
doc = fitz.open(r"E:\AFRAZ\upwork_projects\PDF_COMPARISON\output.pdf")

NEEDLES = ["ÜBERSICHTSCHEMATA", "LEGENDE Datenanbindung"]

top_left_global = None
right_boundary_global = None
for page in doc:
    rects = []
    for needle in NEEDLES:
        # rectangles where this text occurs (case-sensitive by default)
        if needle == "ÜBERSICHTSCHEMATA":
            hits = page.search_for(needle)
            top_left_global = hits[0]
            rects.extend(hits)
        elif needle == "LEGENDE Datenanbindung":
            hits = page.search_for(needle)
            right_boundary_global = hits[0]
            rects.extend(hits)
    
    print("ÜBERSICHTSCHEMATA", top_left_global)
    print("LEGENDE Datenanbindung", right_boundary_global)
    # break
    
    # draw one rectangle per hit
    # rec_draw = fitz.Rect(right_boundary_global.bottom_left.x, top_left_global.top_left.y, top_left_global.top_left.x, right_boundary_global.bottom_left.y)
    # rec_draw = fitz.Rect(right_boundary_global.bottom_left.x, top_left_global.top_left.y, page.rect.height, right_boundary_global.bottom_left.y)
    rec_draw = fitz.Rect(top_left_global.top_left, right_boundary_global.bottom_left.x, page.rect.height)
    print("right_boundary_global.bottom_left.x, top_left_global.top_left.y, top_left_global.top_left.x, right_boundary_global.bottom_left.y")
    print(rec_draw)
    annot = page.add_rect_annot(
        rec_draw
    )   # annotation rectangle
    annot.set_colors(stroke=(1, 0, 0))  # red (RGB 0..1)
    annot.set_border(width=1.5)
    annot.update()

if os.path.exists("abc.pdf"):
    os.remove("abc.pdf")
doc.save("abc.pdf")
doc.close()


ÜBERSICHTSCHEMATA Rect(678.740234375, 36.412960052490234, 906.4224853515625, 56.21283721923828)
LEGENDE Datenanbindung Rect(2013.560546875, 87.33815002441406, 2209.930419921875, 103.18049621582031)
right_boundary_global.bottom_left.x, top_left_global.top_left.y, top_left_global.top_left.x, right_boundary_global.bottom_left.y
Rect(678.740234375, 36.412960052490234, 2013.560546875, 842.7000122070312)


In [144]:
src = fitz.open(r"C:\Users\Lenovo\Downloads\LG13-0101 ff Standortschemen EM\LG13-0101 ff Standortschemen EM\S01\S01_05984_VBA_302500869_LG13-0521_SH_E_E_V00.pdf")
doc = fitz.open()

for src_page in src:  # iterate over input pages
    src_rect = src_page.rect  # source page rect
    w, h = src_rect.br  # save its width, height
    src_rot = src_page.rotation  # save source rotation
    src_page.set_rotation(0)  # set rotation to 0 temporarily
    page = doc.new_page(width=w, height=h)  # make output page
    page.show_pdf_page(  # insert source page
        page.rect,
        src,
        src_page.number,
        rotate=-src_rot,  # use reversed original rotation
    )

doc.ez_save("output.pdf")
doc.close()
src.close()

In [146]:
fitz.open(r"E:\AFRAZ\upwork_projects\PDF_COMPARISON\output.pdf")[0].rotation

0